# Fine-Tune Encoder and Decoder Models


Fine-tuning updates a pretrained model so its representations serve a labeled task. This chapter connects the detector repository's adaptation choices to the capstone: attach a decision head to a pretrained decoder, train it with supervised labels, and update the backbone through full fine-tuning or LoRA. The head returns option scores directly, while the pretrained language model supplies contextual token representations.

## The training interface

A text is tokenized into model-specific token IDs, truncated or chunked to a defined length, and passed through the model. A task-specific classification head may produce logits for two fixed labels. Cross-entropy compares those logits with the recorded class. The optimizer updates selected parameters using training batches, while validation metrics guide checkpoint selection.

The capstone generalizes that interface. The inputs include the state to classify, task instructions, and descriptions of the allowed options. Gold answers appear only in the loss target. Record maximum sequence length, padding policy, label mapping, batch size, optimizer, learning rates, seed, epochs, and checkpoint-selection metric. A different tokenizer or truncation policy defines a different evaluated system.


## Run the worked example

The complete implementation is in the course's `projects/text-decisions` package. Its core functions are displayed below with their executed examples. Reading another repository is optional. From the Watchtower root, install the pinned environment with `uv sync --all-packages`; use its Python kernel to run this notebook. PyTorch 2.14.1, Transformers 5.19.0 and PEFT 0.21.2 are the checked versions.

The supplied data contain six hand-authored training records and four validation records from sentiment and topic tasks. These are mechanics fixtures. The offline run uses tiny randomly initialized Qwen and ModernBERT architectures; the pretrained section uses the same training functions with downloaded weights. Neither a fixture score nor one optimizer step establishes task-transfer quality.

**Setup.** Import the local implementation, set the random seed, and define a source display helper. This displays the actual implementation on the page while keeping one maintained copy in the package.


In [1]:
#| code-fold: false
import inspect
import json
import sys
import textwrap
from pathlib import Path

import torch
from IPython.display import Markdown, display
from torch.nn import functional as F

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "projects/text-decisions").is_dir())
sys.path.insert(0, str(root / "projects/text-decisions/src"))
from text_decisions.model import (
    CandidateHead, DecisionModel, JointHead, candidate_texts, decision_loss,
    joint_text, questions_from, readout, restricted_label_loss, token_span, typed_outputs,
)
from text_decisions.training import (
    adapt_backbone, load_backbone, load_model, make_optimizer, read_records,
    save_model, tiny_backbone, train_step,
)

torch.manual_seed(7)
torch.set_num_threads(2)

def show_source(obj):
    display(Markdown("~~~python\n" + textwrap.dedent(inspect.getsource(obj)) + "\n~~~"))

records = read_records(root / "projects/text-decisions/examples/train.jsonl")
record = records[0]
print([(q.name, q.kind, q.labels) for q in questions_from(record)])


[('sentiment', 'choice', ['negative', 'neutral', 'positive']), ('positive', 'noul', ['false', 'true']), ('rating', 'score', ['0', '1', '2'])]


The first record has three questions with 3, 2 and 3 allowed options. Each has its own normalization and target. `task_id` and `group_id` are split metadata; `answers` supplies supervision and never enters the model input.

## Qwen and ModernBERT change the readout

For token states $\textbf{\textsf{H}}\in\mathbb{R}^{n\times d}$, attention in a backbone layer has the form

$$
\operatorname{Softmax}\left(
\frac{(\textbf{\textsf{H}}\textbf{\textsf{W}}_Q)
(\textbf{\textsf{H}}\textbf{\textsf{W}}_K)^\top}{\sqrt{d_h}}
+\textbf{\textsf{C}}
\right)(\textbf{\textsf{H}}\textbf{\textsf{W}}_V).
$$

Here $d_h$ is one attention head's width and $\textbf{\textsf{C}}$ is the additive attention mask. With zero-based token indices, Qwen's causal mask has $C_{ij}=-\infty$ for $j>i$. Padding is also masked. ModernBERT permits information from both directions, with additional local-window masks in its local attention layers and global attention in other layers. Changing the classification objective preserves these pretrained backbone masks. [Qwen implementation](https://github.com/huggingface/transformers/blob/v5.19.0/src/transformers/models/qwen2/modeling_qwen2.py), [ModernBERT documentation](https://huggingface.co/docs/transformers/model_doc/modernbert).

For the simple candidate scorer, let $a_t$ be 1 for a valid token and 0 for padding. The readouts used in this lab are

$$
\boxed{
\textbf{\textsf{r}}_{\mathrm{Qwen}}
=\textbf{\textsf{h}}_{\max\{t:a_t=1\}},
\qquad
\textbf{\textsf{r}}_{\mathrm{ModernBERT}}
=\frac{\sum_t a_t\textbf{\textsf{h}}_t}{\sum_t a_t}.
}
$$

Qwen's final valid position has access to every preceding token; its first position cannot summarize later input. ModernBERT's mean pool combines bidirectional representations. CLS pooling is another encoder experiment. The implementation finds the last valid position explicitly, so its readout supports left or right padding.

## Begin with a shared candidate scorer

Serialize one sequence per candidate, placing the state and instructions before the candidate description. For a causal decoder, use the final non-padding token representation: that position can attend to the preceding state, instructions, and complete candidate. Apply one shared scalar head and normalize the scores over the candidates belonging to that question. Chapter 05 derives this scoring rule.

This first model makes the supervision easy to inspect. A training example consists of a candidate group, and its loss compares the group's probability distribution with the correct option. Because the same head scores every candidate, the number of labels can vary. Encoding the state repeatedly makes this design expensive for large candidate sets, which motivates the next model.


For each candidate $k$, a small shared MLP computes $z_k=f_\phi(\textbf{\textsf{r}}_k)$. The head's last linear layer has one output regardless of the number of options. The following code implements the readout and scorer.


In [2]:
#| code-fold: false
show_source(readout)
show_source(CandidateHead)
show_source(candidate_texts)


~~~python
def readout(hidden: Tensor, attention_mask: Tensor, pooling: str) -> Tensor:
    """[batch, tokens, features] -> [batch, features], excluding padding."""
    valid = attention_mask.bool()
    if not valid.any(dim=1).all():
        raise ValueError("empty sequence")
    if pooling == "last":
        positions = torch.arange(hidden.size(1), device=hidden.device)
        last = positions.expand_as(valid).masked_fill(~valid, -1).max(dim=1).values
        return hidden[torch.arange(hidden.size(0), device=hidden.device), last]
    if pooling == "mean":
        weights = valid.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1)
    raise ValueError("pooling must be last or mean")

~~~

~~~python
class CandidateHead(nn.Module):
    def __init__(self, hidden_size: int, width: int = 32):
        super().__init__()
        self.scorer = nn.Sequential(nn.Linear(hidden_size, width), nn.GELU(), nn.Linear(width, 1))

    def forward(self, hidden: Tensor, mask: Tensor, pooling: str) -> Tensor:
        return self.scorer(readout(hidden, mask, pooling)).squeeze(-1)

~~~

~~~python
def candidate_texts(state: str, questions: list[Question]) -> list[str]:
    return [
        f"State: {state}\nQuestion: {q.instructions}\n"
        f"Candidate: {label}: {description}\nDecision:"
        for q in questions
        for label, description in zip(q.labels, q.descriptions, strict=True)
    ]

~~~

**Check the causal distinction.** Change the final token while holding the preceding input fixed. Compare the first token's hidden state in the two tiny backbones. Also check that the candidate head returns one score per supplied option.


In [3]:
#| code-fold: false
for family in ("qwen2", "modernbert"):
    backbone, tokenizer, pooling = tiny_backbone(family, records)
    backbone.eval()
    encoded = tokenizer(candidate_texts(record["state"], questions_from(record))[0],
                        return_tensors="pt")
    inputs = {k: v for k, v in encoded.items() if k in {"input_ids", "attention_mask"}}
    changed = {k: v.clone() for k, v in inputs.items()}
    old = int(changed["input_ids"][0, -1])
    changed["input_ids"][0, -1] = 3 + (old + 1) % (len(tokenizer) - 3)
    with torch.no_grad():
        h1 = backbone(**inputs).last_hidden_state
        h2 = backbone(**changed).last_hidden_state
        first_delta = float((h1[0, 0] - h2[0, 0]).abs().max())
        model = DecisionModel(backbone, "candidate", pooling).eval()
        logits = model(record, tokenizer)
    print(f"{family}: pooling={pooling}; first-token change={first_delta:.3e}; "
          f"option counts={[len(z) for z in logits]}")


qwen2: pooling=last; first-token change=0.000e+00; option counts=[3, 2, 3]
modernbert: pooling=mean; first-token change=8.054e-04; option counts=[3, 2, 3]


Qwen's first-token representation stays unchanged because the altered token is in its future. ModernBERT's first-token representation can change through bidirectional attention. This is a representation check; the random models' option scores have no evaluated semantic meaning.

## Pool questions and options from one backbone pass

The joint model serializes the state followed by all questions and allowed options in one sequence. Store the token span of every question and option during serialization. Preserve these spans through padding and truncation, and never truncate away the question schema to accommodate a longer state.

Let $\textbf{\textsf{H}}\in\mathbb{R}^{n\times d}$ be the decoder's final hidden states, where $n$ is the sequence length and $d$ is the backbone feature width. Row $\textbf{\textsf{h}}_t$ represents token position $t$. A span contains the token positions belonging to one question or option. For a nonempty span $S$, its pooled vector is

$$
\textbf{\textsf{u}}_S=\frac{1}{|S|}\sum_{t\in S}\textbf{\textsf{h}}_t.
$$

Pooling turns a variable-length description into a fixed-width vector. The pooled question summarizes its instruction; a pooled option summarizes its label and description in the context available to those tokens. These causal representations have different preceding contexts, so candidate order can influence the result. Shuffle candidate order during training, preserve label-to-position mappings, and test order sensitivity after training.

Project the backbone states into a head width $m$ to form an input memory $\textbf{\textsf{M}}\in\mathbb{R}^{n\times m}$. Project the question and option pools into the same width. In the first joint-head version, each option query combines its projected option vector with the projected instruction for its question.


**Track spans on the complete tokenized input.** The serializer records character ranges as it writes each instruction and option. The fast tokenizer's offset mapping converts them to overlapping token ranges. This avoids estimating a span by separately tokenizing a fragment, whose boundary tokens may differ.


In [4]:
#| code-fold: false
show_source(joint_text)
show_source(token_span)

text, question_chars, option_chars = joint_text(record["state"], questions_from(record))
encoded = tokenizer(text, return_offsets_mapping=True)
offsets = encoded["offset_mapping"]
first_span = token_span(offsets, option_chars[0][0])
print("First option text:", text[slice(*option_chars[0][0])])
print("Its token range:", first_span)
print("Gold answer present in serialized fields:", "answers" in text)


~~~python
def joint_text(state: str, questions: list[Question]) -> tuple[str, list[tuple[int, int]], list[list[tuple[int, int]]]]:
    """Track character spans while writing the input; never include gold answers."""
    text = f"State: {state}\n"
    question_spans = []
    option_spans = []
    for q in questions:
        text += f"\nType: {q.kind}\nQuestion: "
        start = len(text)
        text += q.instructions
        question_spans.append((start, len(text)))
        spans = []
        for label, description in zip(q.labels, q.descriptions, strict=True):
            text += "\nOption: "
            start = len(text)
            text += f"{label}: {description}"
            spans.append((start, len(text)))
        option_spans.append(spans)
        text += "\n"
    return text, question_spans, option_spans

~~~

~~~python
def token_span(offsets: list[tuple[int, int]], span: tuple[int, int]) -> tuple[int, int]:
    """Map a character span to all overlapping tokens of the complete input."""
    start, end = span
    indices = [i for i, (a, b) in enumerate(offsets) if a < end and b > start]
    if not indices:
        raise ValueError("a question or option has no tokens")
    return indices[0], indices[-1] + 1

~~~

First option text: negative: An unfavorable opinion.
Its token range: (23, 29)
Gold answer present in serialized fields: False


## Retrieve evidence with attention

**Cross-attention** uses one set of vectors to ask for information from another set. Here, option queries ask which input token representations are useful for the decision. For one attention head, let $\textbf{\textsf{Q}}\in\mathbb{R}^{K\times m}$ contain the projected queries for $K$ options, and let $\textbf{\textsf{K}},\textbf{\textsf{V}}\in\mathbb{R}^{n\times m}$ be learned key and value projections of the input memory. The attention output is

$$
\textbf{\textsf{A}}
=\operatorname{Softmax}\!\left(
\frac{\textbf{\textsf{Q}}\textbf{\textsf{K}}^\top}{\sqrt m}
\right)\textbf{\textsf{V}}
\in\mathbb{R}^{K\times m}.
$$

The softmax runs across the $n$ memory positions for each query. Its weights determine how much evidence each input position contributes to each option. Mask padding before normalization. PyTorch's multi-head attention repeats this operation in feature subspaces and combines their results.

Use an `nn.MultiheadAttention` module with `batch_first=True` for option-to-memory attention, followed by residual connections, normalization, and a feedforward block. Build each question vector from its instruction, a summary of its evidence-informed options, and an embedding of its type: Choice, Noul, or Score. An `nn.TransformerDecoderLayer` can let these question vectors attend to each other and to the input memory. These layers belong to the classification head; they do not generate answer tokens.

A small shared MLP then scores each question-option pair. A useful starting input concatenates the question vector, option vector, their elementwise product, and their absolute difference. The final linear layer returns one scalar. Group those scalars by question and apply a separate softmax to each question's allowed options.

This is a smaller educational head inspired by Unsloth's [readable per-record implementation](https://github.com/unslothai/unsloth/blob/main/unsloth/models/clef.py). The reference also uses lexical option vectors, a similarity prior, learned scales, and a gated residual score. Record which components the course model includes. The simple scorer and smaller joint head provide comparisons before adding those components. The complete educational head appears below; external source reading is optional.


**Implement the joint head.** This readable version processes one record at a time, so its memory and option lists contain no padding. It retrieves evidence for options, builds question summaries, allows the questions to interact, and returns a separate vector of logits per question. A batched extension must mask token, question and option padding.


In [5]:
#| code-fold: false
show_source(JointHead)


~~~python
class JointHead(nn.Module):
    """One record: span pooling -> evidence attention -> question interaction -> logits."""
    def __init__(self, hidden_size: int, width: int = 32, heads: int = 4):
        super().__init__()
        self.project = nn.Linear(hidden_size, width)
        self.evidence = nn.MultiheadAttention(width, heads, dropout=0.0, batch_first=True)
        self.norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Linear(2 * width, width))
        self.ff_norm = nn.LayerNorm(width)
        self.type_embedding = nn.Embedding(3, width)
        self.interact = nn.TransformerDecoderLayer(width, heads, 2 * width, dropout=0.0, batch_first=True)
        self.scorer = nn.Sequential(nn.Linear(4 * width, width), nn.GELU(), nn.Linear(width, 1))

    def forward(self, hidden: Tensor, question_spans: list[tuple[int, int]],
                option_spans: list[list[tuple[int, int]]], kinds: list[int]) -> list[Tensor]:
        # This readable per-record path has no padded tokens or padded options.
        memory = self.project(hidden)
        question = torch.stack([memory[a:b].mean(dim=0) for a, b in question_spans])
        options = [torch.stack([memory[a:b].mean(dim=0) for a, b in spans]) for spans in option_spans]
        queries = torch.cat([o + q for o, q in zip(options, question, strict=True)])
        evidence, _ = self.evidence(queries[None], memory[None], memory[None], need_weights=False)
        routed = self.norm(queries + evidence[0])
        routed = self.ff_norm(routed + self.ff(routed))
        routed_options = list(routed.split([len(o) for o in options]))
        type_ids = torch.tensor(kinds, device=hidden.device)
        summaries = torch.stack([o.mean(dim=0) for o in routed_options])
        question = question + summaries + self.type_embedding(type_ids)
        # No causal head mask: all questions may interact with all questions/memory.
        question = self.interact(question[None], memory[None])[0]
        logits = []
        for q, o in zip(question, routed_options, strict=True):
            q = q.expand_as(o)
            features = torch.cat([q, o, q * o, (q - o).abs()], dim=-1)
            logits.append(self.scorer(features).squeeze(-1))
        return logits

~~~

## Train probabilities with supervised loss

For one question, let $z_i$ be option $i$'s logit, $p_i$ its normalized probability, and $y_i$ its target probability. There are $K$ allowed options, and the target values sum to one. A hard label uses one target value of one and zeros elsewhere; audited soft labels may supply a distribution. Cross-entropy is

$$
\mathcal{L}_{\mathrm{CE}}
=-\sum_{i=1}^{K}y_i\log p_i.
$$

Compute the loss separately for each question, masking unused option slots in padded batches. Define whether a training record contributes the mean of its question losses or whether questions are averaged across the whole batch. These choices weight records differently when they contain different numbers of questions. Never apply one softmax across unrelated questions.

Choice uses a distribution over mutually exclusive labels. Noul uses two options for each yes/no question and returns its true probability; probabilities from different questions need not sum to one. Score uses a distribution over ordered rubric levels, from which an expected numeric level can be calculated. Adding an ordinal error term is an experiment, rather than a substitute for defining the rubric and testing its meaning.

Begin with cross-entropy. As a controlled extension, add a Brier penalty $\sum_i(p_i-y_i)^2$, or an ordinal penalty for Score questions, and measure the effect on held-out data. Unsloth's [trainer](https://github.com/unslothai/unsloth/blob/main/unsloth/models/decision.py) exposes these choices around a supervised classification loss. The capstone's core training recipe needs no reinforcement learning.


For a hard gold index $g$, expand cross-entropy through the softmax:

$$
\mathcal{L}
=-\log p_g
=-z_g+\log\sum_{k=1}^{K}\exp z_k,
\qquad
\boxed{\frac{\partial\mathcal{L}}{\partial z_k}=p_k-\mathbf{1}[k=g].}
$$

Let $\theta$ be the trainable backbone parameters and $\phi$ the head parameters. The chain rule gives

$$
\nabla_\theta\mathcal{L}
=\sum_k\bigl(p_k-\mathbf{1}[k=g]\bigr)\nabla_\theta z_k.
$$

That gradient reaches the LLM through the decision head and its token representations. The training target is the correct option, so no next-token target is needed for the custom-head path.

For $R$ records, each with $Q_r$ questions, this lab minimizes

$$
\mathcal{L}_{\mathrm{batch}}
=\frac{1}{R}\sum_{r=1}^{R}\frac{1}{Q_r}
\sum_{j=1}^{Q_r}\mathcal{L}_{rj}.
$$

This gives each source record equal weight even when it contains several derived questions. A teacher distribution $t_k$ can replace the one-hot target: $-\sum_k t_k\log p_k$ equals $\operatorname{KL}(t\|p)$ plus the teacher's fixed entropy. It supports supervised soft labels or a distillation experiment; producing a reliable teacher is a separate step.

**Implement the loss.** Hard targets look up a semantic label in the current candidate order. Soft targets name every semantic label, so permuting options preserves the target mapping.


In [6]:
#| code-fold: false
show_source(decision_loss)


~~~python
def decision_loss(logits: list[Tensor], record: dict[str, Any]) -> Tensor:
    """Mean question loss per record. Supports gold labels or audited soft targets."""
    losses = []
    for scores, q in zip(logits, questions_from(record), strict=True):
        answer = record["answers"][q.name]
        if "probabilities" in answer:
            probs = answer["probabilities"]
            if set(probs) != set(q.labels):
                raise ValueError("target distribution must name exactly the allowed labels")
            target = scores.new_tensor([probs[label] for label in q.labels]).float()
            if not torch.isfinite(target).all() or (target < 0).any() or not torch.isclose(target.sum(), target.new_tensor(1.0)):
                raise ValueError("target must be a finite probability distribution")
            losses.append(-(target * F.log_softmax(scores.float(), dim=-1)).sum())
        else:
            target_index = q.labels.index(answer["label"])
            target = torch.tensor([target_index], device=scores.device)
            losses.append(F.cross_entropy(scores.float()[None], target))
    return torch.stack(losses).mean()

~~~

## Warm up the head, then adapt the LLM

First freeze the backbone and train the newly initialized head. Overfit a tiny training subset to check targets, span mapping, option masks, and the loss. A tiny randomly initialized decoder can check these mechanics on a CPU, but says nothing about pretrained task transfer.

Then adapt the pretrained backbone with the decision loss. Full fine-tuning updates its parameters. LoRA adds trainable low-rank updates to selected weight matrices while keeping the base weights fixed. PEFT can construct these adapters, while an explicit PyTorch loop handles the forward pass, loss, backward pass, and optimizer step. Keep the head trainable and give its parameters an explicit optimizer group; the head and adapters may need different learning rates.

Verify that the head and at least one trainable backbone or adapter parameter receive finite gradients and change after an optimizer step. A head-only model is a valid frozen-feature baseline, but does not establish that the LLM was adapted. Avoid disabling gradient recording around the backbone once adapter training begins.

Record optimizer state and the complete artifact needed for inference: tokenizer, serialization and span rules, backbone revision, head configuration and weights, adapter weights when used, type and label mappings, and calibration settings. LoRA reduces the number of trainable parameters and optimizer state; measure total memory and latency rather than assuming reductions in either.

The detector repository also explores MiCA. Inspect its documented configuration when reviewing that experiment; it is a separate adaptation method from the decision-model path developed here.


For a frozen weight $\textbf{\textsf{W}}_0\in\mathbb{R}^{d_{\mathrm{out}}\times d_{\mathrm{in}}}$, LoRA trains $\textbf{\textsf{A}}\in\mathbb{R}^{r\times d_{\mathrm{in}}}$ and $\textbf{\textsf{B}}\in\mathbb{R}^{d_{\mathrm{out}}\times r}$:

$$
\boxed{\textbf{\textsf{W}}_{\mathrm{eff}}
=\textbf{\textsf{W}}_0+\frac{\alpha}{r}\textbf{\textsf{B}}\textbf{\textsf{A}}.}
$$

PEFT inserts these matrices into Qwen's `q_proj` and `v_proj`. ModernBERT's fused `Wqkv` is the target in the encoder variant. Full fine-tuning marks the original backbone parameters trainable instead. Rebuild the optimizer when moving from frozen-head warm-up to backbone adaptation, so its parameter groups include the newly trainable weights.

**Execute the optimizer step.** Gradient accumulation implements the record-mean objective without needing a padded joint-schema batch. The loop records gradients through the backbone, clips them, and applies AdamW.


In [7]:
#| code-fold: false
show_source(adapt_backbone)
show_source(make_optimizer)
show_source(train_step)


~~~python
def adapt_backbone(backbone: Any, method: str) -> Any:
    if method == "lora":
        from peft import LoraConfig, get_peft_model

        # ModernBERT uses a fused QKV projection; Qwen exposes separate projections.
        targets = ["q_proj", "v_proj"] if backbone.config.model_type == "qwen2" else ["Wqkv"]
        return get_peft_model(backbone, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, target_modules=targets))
    backbone.requires_grad_(method == "full")
    return backbone

~~~

~~~python
def make_optimizer(model: DecisionModel, backbone_lr: float, head_lr: float) -> torch.optim.AdamW:
    groups = [{"params": [p for p in model.head.parameters() if p.requires_grad], "lr": head_lr}]
    trainable_backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    if trainable_backbone:
        groups.append({"params": trainable_backbone, "lr": backbone_lr})
    return torch.optim.AdamW(groups, weight_decay=0.0)

~~~

~~~python
def train_step(model: DecisionModel, tokenizer: Any, records: list[dict[str, Any]],
               optimizer: torch.optim.Optimizer, max_length: int = 512) -> float:
    model.train()
    if not any(p.requires_grad for p in model.backbone.parameters()):
        model.backbone.eval()
    optimizer.zero_grad(set_to_none=True)
    total = 0.0
    # Gradient accumulation gives each record equal weight, then averages its questions.
    for record in records:
        logits = model(record, tokenizer, max_length)
        loss = decision_loss(logits, record)
        if not torch.isfinite(loss):
            raise ValueError("non-finite loss")
        (loss / len(records)).backward()
        total += float(loss.detach()) / len(records)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    return total

~~~

The head and backbone use explicit learning-rate groups. A frozen backbone is placed in evaluation mode during warm-up; after adaptation begins, its forward pass remains in the gradient graph. The following check records parameter changes in both components and verifies finite gradients.

**Run all four architecture/head combinations.** The two-layer random models keep this check small and offline. Their losses confirm that optimization runs, not that the models transfer to real tasks.


In [8]:
#| code-fold: false
def audit_step(model, tokenizer, batch, backbone_lr=1e-3, head_lr=1e-2):
    before = {name: p.detach().clone() for name, p in model.named_parameters()
              if p.requires_grad}
    optimizer = make_optimizer(model, backbone_lr, head_lr)
    loss = train_step(model, tokenizer, batch, optimizer)
    changed = {"backbone": 0, "head": 0}
    for name, p in model.named_parameters():
        if name in before and not torch.equal(before[name], p):
            changed[name.split(".")[0]] += 1
        if p.grad is not None:
            assert torch.isfinite(p.grad).all()
    assert changed["backbone"] > 0 and changed["head"] > 0
    return loss, changed

for family in ("qwen2", "modernbert"):
    for head in ("candidate", "joint"):
        backbone, tokenizer, pooling = tiny_backbone(family, records)
        model = DecisionModel(backbone, head, pooling)
        loss_value, changed = audit_step(model, tokenizer, [record])
        print(f"{family}/{head}: loss={loss_value:.4f}; changed tensors={changed}")


qwen2/candidate: loss=0.9605; changed tensors={'backbone': 26, 'head': 4}
qwen2/joint: loss=0.9842; changed tensors={'backbone': 26, 'head': 37}
modernbert/candidate: loss=0.9681; changed tensors={'backbone': 14, 'head': 4}
modernbert/joint: loss=0.9819; changed tensors={'backbone': 14, 'head': 37}


## Fine-tune pretrained Qwen

Replace the random backbone with the pretrained checkpoint and preserve the head, schema and loss. This section defines a complete short run: frozen-head warm-up, LoRA adaptation, inference, saving, reloading, and prediction parity. It uses the fixture only to check mechanics. Run the trainer over audited task data for the capstone's quality and transfer experiment.

The example downloads about 1 GB of checkpoint files and loads float32 weights; allow several GB of RAM. Set `device="cuda"` for a supported GPU. The function runs on CPU too. Its `revision` argument can pin a model commit; the returned run metadata records the resolved revision.

**Run the pretrained pipeline.** The executed example enables `RUN_PRETRAINED`. Set it to `False` for an offline run that skips downloaded weights. The function is defined separately so the same code can be invoked with a pinned checkpoint and a chosen device.


In [9]:
#| code-fold: false
def pretrained_run(checkpoint="Qwen/Qwen2.5-0.5B", revision=None, device="cpu"):
    backbone, tokenizer, pooling = load_backbone(checkpoint, revision)
    resolved = getattr(backbone.config, "_commit_hash", revision)
    model = DecisionModel(backbone, "joint", pooling).to(device)

    model.backbone.requires_grad_(False)
    warmup_optimizer = make_optimizer(model, backbone_lr=2e-5, head_lr=1e-3)
    warmup_loss = train_step(model, tokenizer, [record], warmup_optimizer)

    model.backbone = adapt_backbone(model.backbone, "lora")
    loss_value, changed = audit_step(model, tokenizer, [record],
                                     backbone_lr=2e-5, head_lr=1e-3)
    model.eval()
    with torch.no_grad():
        expected = model(record, tokenizer)
        response = typed_outputs(expected, record)

    directory = root / ".tmp/qwen-course-worked"
    save_model(model, tokenizer, directory, {
        "checkpoint": checkpoint, "revision": resolved,
        "max_length": 512, "adaptation": "lora",
    })
    restored, restored_tokenizer, metadata = load_model(directory, device)
    with torch.no_grad():
        actual = restored(record, restored_tokenizer)
    assert all(torch.allclose(a, b, atol=1e-5)
               for a, b in zip(expected, actual, strict=True))
    return {"checkpoint": checkpoint, "revision": metadata["revision"],
            "warmup_loss": warmup_loss, "adaptation_loss": loss_value,
            "changed_tensors": changed, "reload_matches": True,
            "response": response}

RUN_PRETRAINED = True
if RUN_PRETRAINED:
    print(json.dumps(pretrained_run(), indent=2))
else:
    print("Offline run. Set RUN_PRETRAINED=True to execute pretrained LoRA adaptation.")



{
  "checkpoint": "Qwen/Qwen2.5-0.5B",
  "revision": "060db6499f32faf8b98477b0a26969ef7d8b9987",
  "warmup_loss": 0.987254798412323,
  "adaptation_loss": 0.9081346392631531,
  "changed_tensors": {
    "backbone": 48,
    "head": 37
  },
  "reload_matches": true,
  "response": {
    "sentiment": {
      "label": "positive",
      "probabilities": {
        "negative": 0.29982584714889526,
        "neutral": 0.28376898169517517,
        "positive": 0.4164051413536072
      }
    },
    "positive": {
      "label": "true",
      "probabilities": {
        "false": 0.4828466475009918,
        "true": 0.5171533226966858
      },
      "true_probability": 0.5171533226966858
    },
    "rating": {
      "label": "2",
      "probabilities": {
        "0": 0.30217304825782776,
        "1": 0.3137238025665283,
        "2": 0.38410311937332153
      },
      "expected_level": 1.0819300413131714
    }
  }
}


The pretrained function uses the same `train_step` already exercised above. The recorded run changes 48 backbone tensors belonging to LoRA adapters and 37 head tensors, with finite gradients and matching reloaded predictions. The base backbone weights remain frozen during LoRA. Its initially zero B matrices receive the first update; the A gradients are initially zero because the product BA has B equal to zero. `changed_tensors` measures whether adaptation occurred; its one-record loss and response are not a benchmark. Saved LoRA inference needs both the recorded base revision and adapter weights, plus the independently saved head and tokenizer.

For a multi-epoch run with disjoint validation data, run from the repository root:

~~~bash
uv run --package text-decisions text-decisions train \
  --checkpoint Qwen/Qwen2.5-0.5B --head joint --adaptation lora \
  --train projects/text-decisions/examples/train.jsonl \
  --validation projects/text-decisions/examples/validation.jsonl \
  --epochs 2 --device cuda --output .tmp/qwen-decision

uv run --package text-decisions text-decisions predict \
  --model .tmp/qwen-decision \
  --request projects/text-decisions/examples/request.json
~~~

Use a fresh output directory for each run. Switch to `--head candidate` for the shared-scorer comparison, or `--checkpoint answerdotai/ModernBERT-base --adaptation full` for an encoder run. The CLI shuffles candidate order with semantic targets preserved, rebuilds the optimizer after warm-up, averages losses per record, and selects a checkpoint on validation loss. Replace the fixtures for real task evaluation and keep calibration/test data separate.

## AnyJev: label-token readout and a trainable extension

AnyJev's current `Decider` assigns options single-token answer labels, reads their next-token logits, and renormalizes over those labels. Its L0 readout also averages option rotations and corrects label priors without changing model weights. Its Tacit interface serves checkpoints described as self-distilled; the reviewed checkout provides serving/evaluation, not a training script. These are separate reference paths. [Reviewed levels](https://github.com/nokia-applied-research/AnyJev/blob/d458d13968eec0e2cfa8c81b64629b96cf5cb4a9/docs/levels.md).


Let $\ell_k$ be option $k$'s verified single-token answer ID and $v_{\ell_k}$ its vocabulary logit at the answer position. Restricted normalization cancels the full-vocabulary normalizer:

$$
p_k
=\frac{p_{\mathrm{vocab}}(\ell_k\mid x)}
{\sum_j p_{\mathrm{vocab}}(\ell_j\mid x)}
=\frac{\exp v_{\ell_k}}{\sum_j\exp v_{\ell_j}}.
$$

A supervised extension trains these option scores with the same cross-entropy derived above. The loss supervises relative option probabilities; it does not require the vocabulary-wide answer mass to be high. Inspect that mass separately if reusing a generative interface. This course extension is not a reproduction of Tacit's undisclosed training recipe. [Reviewed label mapping](https://github.com/nokia-applied-research/AnyJev/blob/d458d13968eec0e2cfa8c81b64629b96cf5cb4a9/anyjev/readout.py).

**Implement restricted-label training.** Index the allowed vocabulary columns, then apply cross-entropy. At inference, apply softmax to those same columns.


In [10]:
#| code-fold: false
show_source(restricted_label_loss)

from transformers import Qwen2ForCausalLM
backbone, tokenizer, _ = tiny_backbone("qwen2", records)
lm = Qwen2ForCausalLM(backbone.config)
# The offline fixture assigns two distinct vocabulary IDs as answer labels.
# With a real tokenizer, verify that "A"/"B" each encode to one distinct token.
label_ids = torch.tensor([3, 4])
inputs = tokenizer("State: The movie was wonderful. Question: Sentiment? Answer:",
                   return_tensors="pt")
inputs = {k: v for k, v in inputs.items() if k in {"input_ids", "attention_mask"}}
optimizer = torch.optim.AdamW(lm.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)
vocab_logits = lm(**inputs, use_cache=False).logits[:, -1, :]
loss = restricted_label_loss(vocab_logits, label_ids, torch.tensor([1]))
loss.backward()
optimizer.step()
print(f"Restricted-label training loss: {float(loss.detach()):.4f}")
print("Finite backbone gradient:", bool(torch.isfinite(lm.model.embed_tokens.weight.grad).all()))


~~~python
def restricted_label_loss(vocab_logits: Tensor, label_ids: Tensor, targets: Tensor) -> Tensor:
    """AnyJev-inspired supervised extension: CE over allowed answer-token logits."""
    option_logits = vocab_logits.index_select(-1, label_ids)
    return F.cross_entropy(option_logits.float(), targets)

~~~

Restricted-label training loss: 0.8695
Finite backbone gradient: True


The custom-head path learns an independently initialized scorer. The label-token path reuses a causal LM's vocabulary projection. Both can update the backbone through a decision loss, but they have different output parameters, label-token constraints and prompt assumptions. The offline label-token step uses arbitrary distinct vocabulary IDs to verify the gradient; a real run must validate the answer strings against its tokenizer and render the correct answer position.

Unsloth and AnyJev therefore supply two useful architectural references. The course implementation and its equations are available here; GPU optimizations and external source-code reading are optional.

## Exercises

### 1. Reconstruct a training run

Inspect one detector fine-tuning stage README and its configuration. Reconstruct the data path, tokenizer, maximum length, model checkpoint, optimization settings, selection criterion, and saved files. Explain which files reproduce inference and which resume training. If you run the stage, compare training and validation curves.

### 2. Check the head before scaling

For two short labeled records with different candidate counts, serialize the inputs, print question and option spans, and manually verify the label-to-position mapping. Run a small head and confirm that each question's probabilities sum to one and padded option slots contribute neither probability nor loss. Change candidate order and check that the gold label still maps to the correct position.

### 3. Demonstrate backbone adaptation

Compare a frozen-backbone head-only run with a LoRA or full-fine-tuning run on the same small training subset. Record the trainable parameter groups and gradient norms. Show that the adapted model changes at least one backbone or adapter parameter, then compare validation loss without selecting a model on test results.

### 4. Compare two decision readouts

Using the implementations on this page, explain how the joint head and restricted-label readout produce option scores and where each sends its gradient. For a real tokenizer, verify that two answer labels map to distinct single tokens. Plan a comparison using the same pretrained checkpoint, data and selection budget, and explain why a training-free AnyJev readout is a baseline rather than evidence of backbone adaptation.
